# Imports

In [ ]:
# Standard Library Imports
from datetime import datetime, time
import random
# Third Party Imports
import numpy as np
# Local Imports
from prototypes_and_testing.TreeBasedGenerator import TreeBasedGenerator as Generator, Distribution

# Defining Distributions

In [2]:

class MaleDiseasesDistribution(Distribution):
    def __init__(self):
        super().__init__()
        self.values = ["Disease A", "Disease B", "Disease C", "Disease D"]
        self.probabilities = [0.1, 0.2, 0.3, 0.4]

    def sample(self):
        return str(np.random.choice(self.values, p=self.probabilities))


class FemaleDiseasesDistribution(Distribution):
    def __init__(self):
        super().__init__()
        self.values = ["Disease F", "Disease G", "Disease H", "Disease I"]
        self.probabilities = [0.4, 0.3, 0.2, 0.1]

    def sample(self):
        return str(np.random.choice(self.values, p=self.probabilities))


class GenderDistribution(Distribution):
    def __init__(self):
        super().__init__()
        self.values = ["male", "female"]
        self.probabilities = [0.5, 0.5]

    def sample(self):
        return str(np.random.choice(self.values, p=self.probabilities))


class NumberOfPatientsDuringADayDistribution(Distribution):
    def __init__(self):
        super().__init__()

    def sample(self):
        return int(np.random.normal(50, 10))



class ArrivalTime(Distribution):
    def __init__(self):
        super().__init__()
        self.intervals = [(8, 12), (12, 16), (16, 20), (20, 24)]
        self.probabilities = [0.1, 0.3, 0.4, 0.2]

    def sample(self) -> str:
        start, end = random.choices(self.intervals, weights=self.probabilities, k=1)[0]
        sampled = np.random.uniform(start, end)
        h, rem = divmod(sampled * 3600, 3600)
        m, s = divmod(rem, 60)
        return time(int(h), int(m), int(s)).strftime("%H:%M:%S")

class ABDistribution(Distribution):
    def __init__(self):
        super().__init__()
        self.values = ["A", "B"]
        self.probabilities = [0.5, 0.5]

    def sample(self):
        return str(np.random.choice(self.values, p=self.probabilities))
    
# Instantiate the distributions
num_patients_per_day_dist: Distribution = NumberOfPatientsDuringADayDistribution()
male_diseases_dist: Distribution = MaleDiseasesDistribution()
female_diseases_dist: Distribution = FemaleDiseasesDistribution()
gender_dist: Distribution = GenderDistribution()
arrival_time: Distribution = ArrivalTime()
AB_dist: Distribution = ABDistribution()

# Make Distribution Tree

In [3]:
DistributionTreeNoPatientsPrDay = {
    "number_of_patients": {
        "dist": num_patients_per_day_dist,
        "no_sample_aggregation": lambda x: x,
        "arrival_time": {
            "dist": arrival_time,
            "no_sample_aggregation": lambda x: x[0],
        },
        "gender": {
            "dist": gender_dist,
            "no_sample_aggregation": lambda x: x[0],
            "disease": {
                "dist": {
                    "male": male_diseases_dist,
                    "female": female_diseases_dist
                },
                "no_sample_aggregation": len,
            }
    }
    },

    "other_time_series_data": {
        "dist": num_patients_per_day_dist,
        "no_sample_aggregation": lambda x: x,
        "arrival_time": {
            "dist": arrival_time,
            "no_sample_aggregation": lambda x: x[0],
        },
        "ab": {
            "dist": AB_dist,
            "no_sample_aggregation": lambda x: x[0],
        }
    },
}

# Instantiate and run the generator

In [4]:
generator = Generator(
    distribution_tree=DistributionTreeNoPatientsPrDay, 
    output_format={
        "number_of_patients": ("arrival_time", "gender", "disease"),
        "other_time_series_data": ("arrival_time", "ab")  
    }, 
    output_sorting_function=lambda tup: datetime.strptime(tup[0], "%H:%M:%S").time()
)
output = generator.generate_data()
print(output)

[('08:20:26', 'A'), ('08:23:06', 'male', 'Disease D'), ('08:25:35', 'B'), ('08:29:53', 'A'), ('08:45:43', 'male', 'Disease C'), ('08:49:48', 'A'), ('08:54:13', 'B'), ('09:12:51', 'A'), ('09:16:45', 'female', 'Disease G'), ('09:19:27', 'female', 'Disease G'), ('10:38:50', 'female', 'Disease I'), ('11:16:38', 'B'), ('11:27:54', 'female', 'Disease H'), ('12:19:10', 'female', 'Disease G'), ('12:40:25', 'A'), ('12:47:57', 'female', 'Disease I'), ('13:11:08', 'A'), ('13:18:09', 'A'), ('13:27:49', 'female', 'Disease H'), ('13:38:30', 'A'), ('13:47:33', 'female', 'Disease F'), ('14:01:48', 'male', 'Disease D'), ('14:02:47', 'male', 'Disease C'), ('14:12:15', 'male', 'Disease D'), ('14:14:15', 'B'), ('14:16:09', 'male', 'Disease C'), ('14:18:16', 'female', 'Disease G'), ('14:19:23', 'B'), ('14:22:45', 'female', 'Disease F'), ('14:28:09', 'female', 'Disease F'), ('14:28:22', 'female', 'Disease H'), ('14:28:40', 'A'), ('14:39:58', 'male', 'Disease D'), ('14:41:05', 'male', 'Disease D'), ('14:43:1